In [1]:
import struct
from collections import Counter, defaultdict
from pathlib import Path

from extract_images import JPEG_SOI, collect_images, image_name

CADU_DIR = Path("cadus")

In [2]:
# The 78 byte metadata block holds one big-endian double, 9 bytes into the block (23 bytes into the payload).
# Only four values ever show up: 1.0, 0.0 and +-0.7071067819.
ORIENTATION_OFFSET = 23


def orientation_value(metadata: bytes) -> float:
    return struct.unpack(">d", metadata[ORIENTATION_OFFSET : ORIENTATION_OFFSET + 8])[0]


# Degrees the sun was rotated clockwise relative to the first image, measured by hand on proba2_dump.cadu.
MEASURED_ROTATIONS = {
    "SWAP_20211116T145258Z": 0,
    "SWAP_20211116T145547Z": 0,
    "SWAP_20211116T161230Z": 0,
    "SWAP_20211116T161620Z": 90,
    "SWAP_20211116T161909Z": 90,
    "SWAP_20211116T162034Z": 180,
    "SWAP_20211116T162447Z": 180,
    "SWAP_20211116T162900Z": 270,
    "SWAP_20211116T164843Z": 180,
    "SWAP_20211116T170123Z": 0,
    "SWAP_20211116T170248Z": 0,
    "SWAP_20211116T171115Z": 180,
}

images = {image_name(timestamp): image for timestamp, image in collect_images([CADU_DIR / "proba2_dump.cadu"]).items()}

In [3]:
by_value = defaultdict(set)
print(f"{'image':23s} {'orientation':>12s} {'measured':>8s}")
for name, rotation in MEASURED_ROTATIONS.items():
    value = orientation_value(images[name].metadata)
    by_value[round(value, 4)].add(rotation)
    print(f"{name:23s} {value:12.6f} {rotation:8d}")

print()
for value, rotations in sorted(by_value.items()):
    print(f"orientation {value:+.4f} -> measured rotations {sorted(rotations)}")
print("flag matches the rotations:", all(len(rotations) == 1 for rotations in by_value.values()))

image                    orientation measured
SWAP_20211116T145258Z      -0.707107        0
SWAP_20211116T145547Z      -0.707107        0
SWAP_20211116T161230Z      -0.707107        0
SWAP_20211116T161620Z       1.000000       90
SWAP_20211116T161909Z       1.000000       90
SWAP_20211116T162034Z       0.707107      180
SWAP_20211116T162447Z       0.707107      180
SWAP_20211116T162900Z       0.000000      270
SWAP_20211116T164843Z       0.707107      180
SWAP_20211116T170123Z      -0.707107        0
SWAP_20211116T170248Z      -0.707107        0
SWAP_20211116T171115Z       0.707107      180

orientation -0.7071 -> measured rotations [0]
orientation +0.0000 -> measured rotations [270]
orientation +0.7071 -> measured rotations [180]
orientation +1.0000 -> measured rotations [90]
flag matches the rotations: True


In [4]:
# One orientation value per rotation, as measured above. Not checked against any other dump yet.
ROTATION_BY_ORIENTATION = {-0.7071: 0, 1.0: 90, 0.7071: 180, 0.0: 270}

predicted = Counter()
for path in sorted(CADU_DIR.glob("*.cadu")):
    for image in collect_images([path]).values():
        if image.jpeg.startswith(JPEG_SOI):
            predicted[ROTATION_BY_ORIENTATION[round(orientation_value(image.metadata), 4)]] += 1
print("images per predicted rotation over all dumps:", sorted(predicted.items()))

images per predicted rotation over all dumps: [(0, 98), (90, 83), (180, 54), (270, 89)]
